# Part 10 — Capstone

*CinemaStream — The Forward Deployed Engineer's Handbook*

---

In [ ]:
# ── CinemaStream: one-time setup ──────────────────────────────────────────────
# Run this cell FIRST if you are on Google Colab or a fresh local environment.
# Skip it if you have already cloned the repo and installed requirements.
#
# !pip install -r requirements.txt
# !git clone https://github.com/YOUR_ORG/cinemastream.git
# import os; os.chdir("cinemastream")
# ─────────────────────────────────────────────────────────────────────────────
# Ensure the canonical dataset exists (deterministic; safe to re-run).
try:
    from cinemastream.scripts.generate_data import generate
    generate()
except ModuleNotFoundError:
    print("Run the clone/cd lines above first (Colab), then re-run this cell.")

## Chapters in this notebook

- [Chapter 93: Capstone — The CinemaStream Data Hub](#chapter_93_capstone_the_cinemastream_data_hub)
- [Chapter 93a: Capstone 2 — The Production AI Assistant (MCP + RAG + Observability)](#chapter_93a_capstone_2_the_production_ai_assistant_mcp_rag_observability)
- [Chapter 93b: Capstone 3 — Autonomous Legacy Migration](#chapter_93b_capstone_3_autonomous_legacy_migration)
- [Chapter 93c: Capstone 4 — The Harness-Engineered AI System](#chapter_93c_capstone_4_the_harness_engineered_ai_system)
- [Chapter 94: Using the Worked Solutions — and Assessing Your Readiness](#chapter_94_using_the_worked_solutions_and_assessing_your_readiness)
- [Chapter 95: Main Curriculum Wrap-Up — How to Continue Learning](#chapter_95_main_curriculum_wrap_up_how_to_continue_learning)

---

# Chapter 93: Capstone — The CinemaStream Data Hub

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

### 2.1 The uniform component contract

In [ ]:
from dataclasses import dataclass
from typing import Callable, List

@dataclass
class Probe:
    name: str
    is_ready: Callable[[], bool]
    critical: bool = True          # critical down = stop; non-critical down = degrade

@dataclass
class Status:
    name: str
    ready: bool
    critical: bool

def gate(probes: List[Probe]) -> List[Status]:
    return [Status(p.name, p.is_ready(), p.critical) for p in probes]

def serves(statuses: List[Status]) -> bool:
    """The system serves only if every CRITICAL component is ready."""
    return all(s.ready for s in statuses if s.critical)

### 2.2 The dependency seam

In [ ]:
store_ok = True
def store_ready(): return store_ok
def cache_ready(): return False                  # cache is cold -- degrade, not fail
def api_ready():   return store_ready()          # the seam: API only as ready as the store

probes = [
    Probe("store", store_ready, critical=True),
    Probe("cache", cache_ready, critical=False),
    Probe("api",   api_ready,   critical=True),
]

print("Scenario A -- store healthy:")
statuses = gate(probes)
for s in statuses:
    tag = "OK " if s.ready else ("DOWN" if s.critical else "WARN")
    print(f"  [{tag}] {s.name}")
print(f"  serves? {serves(statuses)}")

### 2.3 Fail-closed: the cascade

In [ ]:
store_ok = False                                 # the store goes down
print("Scenario B -- store fails:")
statuses = gate(probes)
for s in statuses:
    tag = "OK " if s.ready else ("DOWN" if s.critical else "WARN")
    print(f"  [{tag}] {s.name}")
print(f"  serves? {serves(statuses)}")

## 3. CinemaStream in Practice

In [ ]:
import sys; sys.path.insert(0, ".")
from cinemastream.data_hub.hub import build_hub

hub = build_hub()
print(hub.report())

In [ ]:
from cinemastream.data_hub.hub import (
    DataHub, WarehouseComponent, ChurnServiceComponent,
    AssistantComponent, DashboardComponent, DataQualityGate, GateResult,
)

class BrokenGate(DataQualityGate):
    """Replay the Ch053 incident: PH watch_minutes arrive all-NULL overnight."""
    def run(self) -> GateResult:
        real = super().run()
        results = [(n, False if "all-NULL" in n else ok, d) for n, ok, d in real.results]
        return GateResult(passed=all(o for _, o, _ in results), results=results)

warehouse = WarehouseComponent(BrokenGate())
hub = (DataHub()
       .register(warehouse)
       .register(ChurnServiceComponent())
       .register(AssistantComponent())
       .register(DashboardComponent(warehouse)))
print(hub.report())

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
def reporting_ready(): return store_ready() and api_ready()
probes.append(Probe("reporting", reporting_ready, critical=False))

---

# Chapter 93a: Capstone 2 — The Production AI Assistant (MCP + RAG + Observability)

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

### 2.1 The router and the trace

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class Span:
    name: str; ms: int

@dataclass
class Trace:
    route: str = "?"
    spans: List[Span] = field(default_factory=list)
    @property
    def latency(self): return sum(s.ms for s in self.spans)

TOOL_WORDS = {"add", "sum", "multiply"}
FAQ_WORDS  = {"hours", "refund", "contact"}

def route(q):
    w = set(q.lower().split())
    if w & TOOL_WORDS: return "TOOL"
    if w & FAQ_WORDS:  return "FAQ"
    return "REFUSE"

### 2.2 The instrumented handler

In [ ]:
FAQ = {"hours": "Open 9 to 5.", "refund": "Refunds within 14 days.",
       "contact": "Email support@x.io."}

def handle(q):
    t = Trace(); t.spans.append(Span("route", 5)); kind = route(q); t.route = kind
    if kind == "TOOL":
        t.spans.append(Span("tool", 12))
        nums = [int(x) for x in q.split() if x.isdigit()]
        ans = f"Result: {sum(nums)}"
    elif kind == "FAQ":
        t.spans.append(Span("retrieve", 20))
        key = next(k for k in FAQ if k in q.lower()); ans = FAQ[key]
    else:
        ans = "Out of scope -- I can't help with that."
    t.spans.append(Span("generate", 30))
    return ans, t

for q in ["add 2 and 3", "what are your hours", "tell me a joke"]:
    ans, t = handle(q)
    print(f"[{t.route}] {q!r} -> {ans}  ({t.latency}ms)")

### 2.3 The eval gate

In [ ]:
GOLDEN = [("add 2 and 3", "TOOL"), ("what are your hours", "FAQ"),
          ("refund policy?", "FAQ"), ("tell me a joke", "REFUSE")]
passed = sum(1 for q, exp in GOLDEN if route(q) == exp)
print(f"Eval: {passed}/{len(GOLDEN)} routes correct")

## 3. CinemaStream in Practice — On Loan at FilmiBox

In [ ]:
import sys; sys.path.insert(0, ".")
from filmibox.assistant.assistant import ask

demos = [
    ("support_agent", "double charge on siti@example.com?"),
    ("new_hire",      "how do I get VPN access?"),
    ("support_agent", "what are our partner contract terms?"),
    ("new_hire",      "double charge on ravi@example.com?"),
]
for role, q in demos:
    r = ask(role, q)
    print(f"[{role}] {q}")
    print(f"   route={r.trace.route} | prompt={r.trace.prompt_version} | "
          f"latency={r.trace.latency_ms}ms | cost=S${r.trace.cost_sgd}")
    print(f"   -> {r.answer}")

In [ ]:
from filmibox.assistant.assistant import run_evals, percentile

passed, total, traces = run_evals()
print(f"Routing/refusal accuracy: {passed}/{total}")

lats = [t.latency_ms for t in traces]
routes = {}
for t in traces:
    routes[t.route] = routes.get(t.route, 0) + 1
print(f"requests={len(traces)} | p50={percentile(lats,50)}ms | "
      f"p95={percentile(lats,95)}ms | total_cost=S${round(sum(t.cost_sgd for t in traces),5)}")
print("route mix: " + ", ".join(f"{k}={v}" for k, v in sorted(routes.items())))

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
def route(q):
    w = set(q.lower().split())
    if w & {"urgent", "complaint"}: return "ESCALATE"   # FIRST -- highest priority
    if w & TOOL_WORDS: return "TOOL"
    if w & FAQ_WORDS:  return "FAQ"
    return "REFUSE"

---

# Chapter 93b: Capstone 3 — Autonomous Legacy Migration

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

### 2.1 Extract structured fields from free text

In [ ]:
import re
from dataclasses import dataclass
from typing import Optional, Tuple

LEGACY = [
    "Order 101: city=Delhi   amount=450",
    "order 102 - CITY DELHI - AMT 500",
    "Ord 103 :: mumbai :: amount  700",
    "corrupt row ### no fields ???",        # -> quarantine
]

CITY = re.compile(r"(?:city[=\s:]+|CITY\s+)([A-Za-z]+)", re.IGNORECASE)
AMT  = re.compile(r"(?:amount|amt)[=\s]+(\d+)", re.IGNORECASE)

@dataclass
class Clean:
    city: str; amount: int

def structure(row: str) -> Tuple[Optional[Clean], Optional[str]]:
    c, a = CITY.search(row), AMT.search(row)
    if not (c and a):                       # a field is missing -> quarantine
        return None, f"unparseable: {row[:24]!r}"
    return Clean(c.group(1).title(), int(a.group(1))), None

### 2.2 Route every row — structure or quarantine

In [ ]:
clean, quarantine = [], []
for row in LEGACY:
    rec, q = structure(row)
    (clean if rec else quarantine).append(rec or q)

print("Structured:")
for r in clean:
    print(f"  {r.city:8s} {r.amount}")
print("Quarantined (never dropped):")
for q in quarantine:
    print(f"  {q}")
print(f"\n{len(LEGACY)} ingested -> {len(clean)} structured, {len(quarantine)} quarantined")

```
Output (cont.):
4 ingested -> 2 structured, 2 quarantined
```

## 3. CinemaStream in Practice — On Loan at FilmiBox

In [ ]:
import sys; sys.path.insert(0, ".")
from filmibox.migration.migrate import migrate

result = migrate()
print("\n".join(result.report.lines()))

In [ ]:
from fastapi.testclient import TestClient
from filmibox.migration.migrate import build_api

client = TestClient(build_api(result))

r1 = client.get("/customer/ravi@example.com")
print(f"GET /customer/ravi@example.com -> {r1.status_code} {r1.json()}")

r2 = client.get("/customer/ghost@example.com")
print(f"GET /customer/ghost@example.com -> {r2.status_code} (not found)")

r3 = client.get("/search", params={"q": "refunds basic"})
print(f"GET /search?q=refunds basic -> {[h['email'] for h in r3.json()['hits']]}")

r4 = client.get("/migration/report")
print(f"GET /migration/report -> {r4.json()}")

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
CITY = re.compile(r"(?:city[=\s:]+|CITY\s+|::\s*)([A-Za-z]+)", re.IGNORECASE)

---

# Chapter 93c: Capstone 4 — The Harness-Engineered AI System

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

### 2.1 The gate and the verdict

In [ ]:
from dataclasses import dataclass
from enum import Enum
from typing import List

class Status(str, Enum):
    PASS = "PASS"; WARN = "WARN"; BLOCK = "BLOCK"

@dataclass
class GateResult:
    name: str; status: Status; detail: str

def lint_gate(art):   return GateResult("lint", Status.PASS if art["lints"] else Status.BLOCK, "style")
def test_gate(art):   return GateResult("test", Status.PASS if art["tests_pass"] else Status.BLOCK, f"{art['tests_pass']}")
def budget_gate(art):
    over = art["cost"] > art["budget"]; near = art["cost"] > 0.8 * art["budget"]
    s = Status.BLOCK if over else (Status.WARN if near else Status.PASS)
    return GateResult("budget", s, f"cost={art['cost']} budget={art['budget']}")

GATES = [lint_gate, test_gate, budget_gate]

def verdict(results: List[GateResult]) -> Status:
    if any(r.status == Status.BLOCK for r in results): return Status.BLOCK
    if any(r.status == Status.WARN for r in results):  return Status.WARN
    return Status.PASS

### 2.2 Running the suite

In [ ]:
def run(art):
    results = [g(art) for g in GATES]
    for r in results:
        print(f"  [{r.status.value:5s}] {r.name}: {r.detail}")
    v = verdict(results)
    print(f"  VERDICT: {v.value} (deploy {'BLOCKED' if v == Status.BLOCK else 'ALLOWED'})")

print("Artifact A -- healthy:")
run({"lints": True, "tests_pass": True, "cost": 70, "budget": 100})
print("\nArtifact B -- tests fail:")
run({"lints": True, "tests_pass": False, "cost": 95, "budget": 100})

## 3. CinemaStream in Practice

In [ ]:
import sys; sys.path.insert(0, ".")
from cinemastream.harness.harness import run_harness, report

print(report(run_harness()))

In [ ]:
from cinemastream.harness.harness import (
    run_harness, report, golden_eval_gate, adversarial_gate,
    observability_gate, cost_gate, GateResult, Status,
)

def architecture_gate_regressed() -> GateResult:
    # An AI agent "simplified" the assistant and dropped the access-control map.
    return GateResult("architecture", Status.BLOCK,
                      "missing rules: ['access control present']")

run = run_harness([architecture_gate_regressed, golden_eval_gate,
                   adversarial_gate, observability_gate, cost_gate])
print(report(run))

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
def security_gate(art):
    if art.get("vulns", 0) == 0:
        return GateResult("security", Status.PASS, "no known vulns")
    s = Status.WARN if art.get("in_grace_period") else Status.BLOCK
    return GateResult("security", s, f"{art['vulns']} vuln(s)")

---

# Chapter 94: Using the Worked Solutions — and Assessing Your Readiness

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

### 2.1 The check harness

In [ ]:
from dataclasses import dataclass
from typing import Callable, List, Any

@dataclass
class Case:
    args: tuple
    expected: Any

def check(fn: Callable, cases: List[Case]) -> int:
    passed = 0
    for i, c in enumerate(cases, 1):
        got = fn(*c.args)
        ok = got == c.expected
        passed += ok
        extra = "" if ok else f"  (expected {c.expected!r})"
        print(f"  [{'PASS' if ok else 'FAIL'}] case {i}: got {got!r}{extra}")
    print(f"  -> {passed}/{len(cases)} passed")
    return passed

### 2.2 Using it on an attempt

In [ ]:
def evens_doubled(nums):
    return [n * 2 for n in nums if n % 2 == 0]

print("Self-check: evens_doubled")
check(evens_doubled, [
    Case(([1, 2, 3, 4],), [4, 8]),
    Case(([],), []),
    Case(([5, 7],), []),
])

## 3. CinemaStream in Practice

In [ ]:
# M1 (Python): churn rate from a list of user dicts, rounded to 3 dp.
def churn_rate(users):
    if not users:
        return 0.0
    return round(sum(u["churned"] for u in users) / len(users), 3)

# M3 (data logic): the top genre by COMPLETED watch count.
def top_genre(events):
    from collections import Counter
    c = Counter(e["genre"] for e in events if e["completed"])
    return c.most_common(1)[0][0] if c else None

# M9 (ML metric): precision for the positive class from confusion counts.
def precision(tp, fp):
    return round(tp / (tp + fp), 3) if (tp + fp) else 0.0

print("M1 -- churn_rate:")
check(churn_rate, [
    Case(([{"churned": 1}, {"churned": 0}, {"churned": 0}],), 0.333),
    Case(([],), 0.0),
])
print("M3 -- top_genre:")
check(top_genre, [
    Case(([{"genre": "Drama", "completed": True},
           {"genre": "Drama", "completed": True},
           {"genre": "Action", "completed": False}],), "Drama"),
    Case(([{"genre": "Action", "completed": False}],), None),
])
print("M9 -- precision:")
check(precision, [
    Case((3, 11), 0.214),     # Ch073 tuned churn model: tp=3, fp=11
    Case((0, 0), 0.0),
])

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
cases = [
    Case(("a@b.com",), True),       # ordinary valid
    Case(("a@b@c",), False),        # two @ -> invalid
    Case(("@b.com",), False),       # empty local part
    Case(("ab.com",), False),       # no @ at all
    Case(("a@",), False),           # empty domain part
]

In [ ]:
def monthly_arpu(subscriptions):
    if not subscriptions:
        return 0.0
    return round(sum(s["amount_sgd"] for s in subscriptions) / len(subscriptions), 2)

check(monthly_arpu, [
    Case(([{"amount_sgd": 19.9}, {"amount_sgd": 12.9}],), 16.4),
    Case(([{"amount_sgd": 19.9}],), 19.9),
    Case(([],), 0.0),     # empty input: no subscribers this slice
])

---

# Chapter 95: Main Curriculum Wrap-Up — How to Continue Learning

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

### 2.1 The T-Shaped Skill Map

### 2.2 How to Stay Current

In [ ]:
from dataclasses import dataclass, field
from datetime import date
from typing import List


@dataclass
class LearningEntry:
    topic: str
    built: str          # what you made
    gap_closed: str     # what you can do now that you couldn't
    source: str         # paper / project / community / colleague
    date_added: date = field(default_factory=date.today)


Q3_2026: List[LearningEntry] = [
    LearningEntry(
        topic="Qdrant vector store",
        built="Replaced movie_rag.py FAISS index with Qdrant collection",
        gap_closed="Can deploy a persistent, filterable vector store in production",
        source="Qdrant docs + Chapter 085b CinemaStream patterns",
        date_added=date(2026, 7, 15),
    ),
    LearningEntry(
        topic="LangGraph",
        built="Multi-agent FilmiBox assistant with tool routing and memory",
        gap_closed="Can build stateful agentic workflows, not just single-turn chains",
        source="LangGraph docs + MLOps Community Slack",
        date_added=date(2026, 7, 28),
    ),
]

for e in Q3_2026:
    print(f"[{e.date_added}] {e.topic}")
    print(f"  Built:       {e.built}")
    print(f"  Gap closed:  {e.gap_closed}")
    print(f"  Source:      {e.source}")
    print()

### 2.3 The Teach-It Principle

### 2.4 Books Worth Reading Next

### 2.5 The Portfolio as a Living Artifact

In [ ]:
# Extensions to the cinemastream/ repo that signal genuine continued growth.
# Each one is a portfolio story — not just "I added a feature" but
# "I noticed a gap and closed it, and here's what I learned."

PORTFOLIO_EXTENSIONS = [
    {
        "extension": "Replace FAISS with Qdrant in movie_rag.py",
        "file": "cinemastream/ml/content_tagging/movie_rag.py",
        "signals": "Can operate a persistent vector store; understands filtering at index level",
        "chapter_foundation": "Ch085b — Vector Database Decision Guide",
    },
    {
        "extension": "Add real LLM API calls to filmibox/assistant/assistant.py",
        "file": "filmibox/assistant/assistant.py",
        "signals": "Understands prompt versioning, cost tracking, real API latency vs mocked",
        "chapter_foundation": "Ch093a — Production AI Assistant",
    },
    {
        "extension": "Deploy serve.py to AWS Lambda + add CloudWatch latency alert",
        "file": "cinemastream/ml/churn/serve.py",
        "signals": "Can deploy and monitor a model endpoint end-to-end on cloud infrastructure",
        "chapter_foundation": "Ch077 — FastAPI + Ch052 — Monitoring",
    },
    {
        "extension": "Add a latency regression gate to cinemastream/harness/harness.py",
        "file": "cinemastream/harness/harness.py",
        "signals": "Understands the harness as extensible; can encode a new constraint as a gate",
        "chapter_foundation": "Ch093c — The Harness-Engineered AI System",
    },
    {
        "extension": "Write a nightly Airflow DAG that retrains the churn model",
        "file": "cinemastream/pipelines/watch_events_dag.py",
        "signals": "Can close the loop from pipeline to model to deployment automatically",
        "chapter_foundation": "Ch047 — Real Airflow DAG + Ch067–077 — Applied ML",
    },
    {
        "extension": "Swap the Data Hub MRR forecast to TimesFM zero-shot",
        "file": "cinemastream/streamlit_app/app.py",
        "signals": "Can evaluate a foundation model against a classical baseline and keep the Ch077b backtest as the judge",
        "chapter_foundation": "Ch077c — Foundation Models for Structured Data",
    },
]

for ext in PORTFOLIO_EXTENSIONS:
    print(f"Extension: {ext['extension']}")
    print(f"  File:      {ext['file']}")
    print(f"  Signals:   {ext['signals']}")
    print(f"  Foundation: {ext['chapter_foundation']}")
    print()

## 3. CinemaStream in Practice

In [ ]:
from dataclasses import dataclass
from typing import List


@dataclass
class CareerPath:
    title: str
    what_you_do: str
    strongest_chapters: List[str]
    depth_spike_needed: str
    typical_next_role: str
    honest_warning: str


PATHS = [
    CareerPath(
        title="ML Engineer",
        what_you_do="Deploy, monitor, retrain, and evaluate models at production scale",
        strongest_chapters=["Ch067-077 (churn model end-to-end)",
                            "Ch077 (FastAPI serve.py)",
                            "Ch075 (MLflow experiment tracking)",
                            "Ch093c (harness gates)"],
        depth_spike_needed="Kubernetes, KubeFlow or Vertex AI, feature store at scale, "
                           "distributed training (PyTorch DDP or FSDP)",
        typical_next_role="Senior ML Engineer → ML Platform Lead → Head of ML",
        honest_warning="The gap between 'deployed one model' and 'owns the ML platform' "
                       "is operational depth: on-call, rollback, retraining automation.",
    ),
    CareerPath(
        title="Data Engineer",
        what_you_do="Design and operate the pipelines that feed every downstream consumer",
        strongest_chapters=["Ch046-055 (Airflow, CDC, contracts, observability)",
                            "Ch049 (dbt)",
                            "Ch050 (cloud warehouses)",
                            "Ch057 (incident response)"],
        depth_spike_needed="Apache Spark or Flink, Kafka at scale, "
                           "cloud-native pipeline tools (Dataflow, Glue, dbt Cloud)",
        typical_next_role="Senior Data Engineer → Data Platform Architect → Staff Engineer",
        honest_warning="The move from 'runs Airflow' to 'owns the data platform' "
                       "requires reliability engineering depth most data engineers skip.",
    ),
    CareerPath(
        title="Analytics Engineer",
        what_you_do="Sit at the intersection of data engineering and business intelligence",
        strongest_chapters=["Ch032-041 (SQL to window functions)",
                            "Ch049 (dbt)",
                            "Ch060-063 (Streamlit, UX, role-based views)",
                            "Ch056 (FinOps for data)"],
        depth_spike_needed="Advanced dbt (macros, packages, semantic layer), "
                           "BI tools (Metabase, Looker, Superset), "
                           "stakeholder communication at exec level",
        typical_next_role="Analytics Engineering Lead → Head of Analytics → CDO",
        honest_warning="Business translation is the hard part — not the SQL. "
                       "Ch086-092 (consulting module) is directly applicable.",
    ),
    CareerPath(
        title="FDE / AI Solutions Engineer",
        what_you_do="Embed with client teams to ship AI-native data systems end-to-end",
        strongest_chapters=["Ch086-092 (FilmiBox engagement)",
                            "Ch085-085k (RAG, evals, security, harness)",
                            "Ch093a-093c (production AI capstones)",
                            "Ch064-066 (prompt engineering, LLM workflows)"],
        depth_spike_needed="Production LLM systems (eval harnesses, guardrails, "
                           "cost controls), client communication, discovery-to-delivery",
        typical_next_role="Senior FDE → Principal Solutions Architect → "
                          "Head of AI Engineering",
        honest_warning="The productization feedback loop (Ch086 definition) "
                       "is what makes FDE different from consulting. "
                       "Without it, you are an expensive contractor.",
    ),
]

for p in PATHS:
    print(f"\n{'='*55}")
    print(f"  {p.title}")
    print(f"  {p.what_you_do}")
    print(f"  Key chapters: {p.strongest_chapters[0]} ...")
    print(f"  Need to add:  {p.depth_spike_needed[:60]}...")
    print(f"  Path:         {p.typical_next_role}")
    print(f"  Warning:      {p.honest_warning[:60]}...")

## 4. Pitfalls & Pro Tips

## 5. Exercises